In [1]:
import os, sys
from pathlib import Path
import cortex
import nibabel as nb
import numpy as np
import matplotlib.colors as colors
import matplotlib.pyplot as pl
import time
import platform
import pickle
import pandas as pd
from matplotlib.colors import Normalize
from prf_expect.utils import io

In [2]:
surprise_analysis_type = 'TRMI-type1'

In [3]:
# Define paths and data exp parameters
settings = io.load_settings()
data_dir = Path(settings["general"]["data_dir"], "data")
tasks = settings["design"]["tasks"]
space = settings["mri"]["space"]

Loading settings from /Users/dionysus/Library/CloudStorage/OneDrive-Personal/Workbench/pRF_expect_pub/pRF_expect_analysis/prf_expect/settings.yml


In [4]:
subjects = ["sub-001", "sub-002", "sub-004", "sub-005", "sub-007", "sub-009", "sub-012"]

glm_rsq = []
params_rsqs = []
for subject in subjects:
    analysis_result_dir = f"/Users/dionysus/Downloads/data/derivatives/prf_data/{subject}/ses-1/{surprise_analysis_type}"
    analysis_result_dir = Path(analysis_result_dir)
    tsv_name = Path.joinpath(
        data_dir,
        "derivatives",
        "prf_data",
        subject,
        "ses-1",
        "prf_fits",
        "prf_params",
        f"{subject}_ses-1_final-fit_space-{space}_model-norm_stage-iter_desc-prf_params.tsv",
    )
    params = pd.read_csv(tsv_name, sep="\t")
    params_rsq_sub = params["r2"].values
    params_rsq_sub[~np.isfinite(params_rsq_sub)] = 0.0
    params_rsq_sub = np.clip(params_rsq_sub, 0, None)  # enforce non-negative weights
    params_rsq_sub[params_rsq_sub == 1.0] = 0

    fn = f"{subject}_ses-1_space-fsaverage_surprise-viol_rsq.npy"
    glm_rsq_fn = analysis_result_dir / fn
    glm_rsq_sub = np.load(glm_rsq_fn, allow_pickle=True)
    glm_rsq.append(glm_rsq_sub)
    params_rsqs.append(params_rsq_sub)

glm_rsq = np.array(glm_rsq, dtype=float)
glm_rsq[~np.isfinite(glm_rsq)] = 0.0

params_rsqs = np.array(params_rsqs, dtype=float)
params_rsqs[~np.isfinite(params_rsqs)] = 0.0
params_rsqs = np.clip(params_rsqs, 0, None)

# Weighted mean per vertex; if all weights are zero, set output to 0 for that vertex.
weight_sum = np.sum(params_rsqs, axis=0)
weighted_sum = np.sum(glm_rsq * params_rsqs, axis=0)
glm_rsq = np.divide(weighted_sum, weight_sum, out=np.zeros_like(weight_sum), where=weight_sum > 0)
params_rsqs_mean = np.mean(params_rsqs, axis=0)

In [5]:
cwd = os.getcwd()
figure_result_dir = Path(cwd).parent / "figures"
print('Running on {}'.format(platform.node()))
print('Current deriv folder is {}'.format(analysis_result_dir))

print('cortex.database.default_filestore: {}'.format(cortex.database.default_filestore))
print('cortex.options.usercfg: {}'.format(cortex.options.usercfg))

Running on Smoldering-Corpse-Bar.local
Current deriv folder is /Users/dionysus/Downloads/data/derivatives/prf_data/sub-012/ses-1/TRMI-type1
cortex.database.default_filestore: /Users/dionysus/Documents/pycortex/db
cortex.options.usercfg: /Users/dionysus/Library/Application Support/pycortex/options.cfg


In [6]:
subject = "fsaverage-vis"

In [7]:
static_imgs = False
web_view = True

In [8]:
def con_weighted_dispcx(subject, data, param_rsq, cmap='seismic', vmin=-0.3, vmax=0.3, vmin2=0, vmax2=1, rsq_thre=0.25):
    curv = cortex.db.get_surfinfo(subject)
    # Adjust curvature contrast / color. Alternately, you could work
    # with curv.data, maybe threshold it, and apply a color map. 
    curv.vmin = -1
    curv.vmax = 1
    curv.cmap = 'gray'
    curv.data = curv.data * .75 + 0.6
    # curv.data = curv.data * .50
    norm2 = Normalize(vmin2, vmax2)
    # curv = cortex.Vertex(curv.data, subject, vmin=-1,vmax=1,cmap='gray')
    # Create some display data

    # # normalize the range of param_rsq to 0 to 1
    # if param_rsq.max()-param_rsq.min()>0:
    #     param_rsq = (param_rsq-param_rsq.min())/(param_rsq.max()-param_rsq.min()-0.2)
    # else:
    #     pass
    # alpha = np.clip(norm2(param_rsq), 0, 1)
    alpha = np.clip(norm2(param_rsq), 1, 1)
    alpha[param_rsq<rsq_thre] = 0
    vx = cortex.Vertex(data, subject, cmap=cmap, vmin=vmin, vmax=vmax, )

    # Map to RGB
    vx_rgb = np.vstack([vx.raw.red.data, vx.raw.green.data, vx.raw.blue.data])
    curv_rgb = np.vstack([curv.raw.red.data, curv.raw.green.data, curv.raw.blue.data])

    
    # alpha = param_rsq
    alpha = alpha.astype(np.float32)

    # Alpha mask
    display_data = vx_rgb * alpha + curv_rgb * (1 - alpha)
    # fake_curv_rgb = np.zeros_like(curv_rgb)
    # display_data = vx_rgb * alpha + fake_curv_rgb * (1 - alpha)
    # display_data /= 255
    return display_data

In [9]:
def Vertex2D_fix(data1, data2, subject, cmap, vmin, vmax, vmin2, vmax2, roi_borders=None):
    #this provides a nice workaround for pycortex opacity issues, at the cost of interactivity    
    # Get curvature
    curv = cortex.db.get_surfinfo(subject)
    # Adjust curvature contrast / color. Alternately, you could work
    # with curv.data, maybe threshold it, and apply a color map. 
    
    #standard
    curv.data = curv.data * .75 +0.1
    #alternative
    #curv.data = np.sign(curv.data) * .25
    #HCP adjustment
    #curv.data = curv.data * -2.5# 1.25 +0.1

    
    curv = cortex.Vertex(curv.data, subject, vmin=-1,vmax=1,cmap='gray')
    
    norm2 = Normalize(vmin2, vmax2)   
    
    vx = cortex.Vertex(data1, subject, cmap=cmap, vmin=vmin, vmax=vmax)
    
    # Map to RGB
    vx_rgb = np.vstack([vx.raw.red.data, vx.raw.green.data, vx.raw.blue.data])
    
    curv_rgb = np.vstack([curv.raw.red.data, curv.raw.green.data, curv.raw.blue.data])

    
    # Pick an arbitrary region to mask out
    # (in your case you could use np.isnan on your data in similar fashion)
    alpha = np.clip(norm2(data2), 0, 1)

    # Alpha mask
    display_data = (curv_rgb * (1-alpha)) + vx_rgb * alpha

    display_data /= 255


    #print(display_data.min())
    #print(display_data.max())
    
    if roi_borders is not None:
        display_data[:,roi_borders.astype('bool')] = 0#255-display_data[:,roi_borders.astype('bool')]#0#255
    
    # Create vertex RGB object out of R, G, B channels
    return cortex.VertexRGB(*display_data, subject)  

In [10]:
np.nanmax(glm_rsq)

np.float64(0.45200567002299313)

In [11]:
rsq_alpha = np.ones_like(glm_rsq)
rsq_alpha[params_rsqs_mean<0.1] = 0

display_rsq = con_weighted_dispcx(
    subject, 
    glm_rsq, 
    rsq_alpha, 
    cmap='inferno', 
    vmin=0.0, 
    vmax=0.45,
    rsq_thre=0.1
)
rsq_cx = cortex.VertexRGB(
    *display_rsq, 
    subject,
)

    
if web_view:
    print("creating web view")
    for d, nm, depth in zip([rsq_cx.raw,], 
                            ["cx_viol_rsq"], 
                            [0, ]):
        ds = cortex.Dataset(**{nm: d})
        handle = cortex.webgl.show(data=ds, recache=True, labels_visible=(), overlays_visible=('rois',))
        file_pattern = "{base}_{view}_{nm}.png"
        time.sleep(20.0)
        # projection parameters
        basic = dict(
            radius=300, depth=depth, specularity=0, unfold=0.5, contrast=0
        )  # projection=['orthographic'],
        # different views available, more views can be added  and the
        # existing list can be removed
        views = dict(
            myview=dict(altitude=94, azimuth=189, pivot=0),
            # lateral=dict(altitude=90.5, azimuth=181, pivot=180),
            # medial=dict(altitude=90.5, azimuth=0, pivot=180),
            # front=dict(altitude=90.5, azimuth=0, pivot=0),
            # back=dict(altitude=90.5, azimuth=181, pivot=0),
            # top=dict(altitude=0, azimuth=180, pivot=0),
            # bottom=dict(altitude=180, azimuth=0, pivot=0),
        )
        # utility functions to set the different views
        prefix = dict(
            altitude="camera.",
            azimuth="camera.",
            pivot="surface.{subject}.",
            radius="camera.",
            unfold="surface.{subject}.",
            depth="surface.{subject}.",
            specularity="surface.{subject}.",
            contrast="surface.curvature.",
        )
        _tolists = lambda p: {prefix[k] + k: [v] for k, v in p.items()}
        _combine = lambda a, b: (lambda c: [c, c.update(b)][0])(dict(a))
        # Save images by iterating over the different views and surfaces
        for view, vparams in views.items():
            # Combine basic, view, and surface parameters
            params = _combine(basic, vparams)
            # Set the view
            handle._set_view(**_tolists(params))
            # Save image
            if "viol" in nm or "TRMI" in nm:
                filename = file_pattern.format(base=subject+f"_{surprise_analysis_type}", view=view, nm=nm)
            else:
                filename = file_pattern.format(base=subject, view=view, nm=nm)

            output_path = os.path.join(
                figure_result_dir, filename
            )
            handle.getImage(output_path, size=(3840, 2160))
            # the block below trims the edges of the image:
            # wait for image to be written
            while not os.path.exists(output_path):
                pass
            time.sleep(1.5)
            # try:
            #     import subprocess
            #     subprocess.call(['convert', '-trim', output_path, output_path])
            # except:
            #     pass
        # Close the window!
        handle.close()

/Users/dionysus/anaconda3/envs/prfexpectpub/lib/python3.13/site-packages/numpy/lib/_function_base_impl.py:4859: UserWarning: Warning: 'partition' will ignore the 'mask' of the MaskedArray.
  arr.partition(


creating web view
Generating new ctm file...
wm
wm
inflated
inflated
Started server on port 47993
Unknown parameter surface.curvature.contrast!


Stopping server
